In [16]:
from langgraph.graph import StateGraph, START, END
from langchain_groq import ChatGroq
from dotenv import load_dotenv
from typing import TypedDict, Annotated
from langchain_core.messages import BaseMessage,HumanMessage
from langgraph.graph.message import add_messages
import os
import requests
import random
from langchain_core.tools import tool
from langgraph.prebuilt import tools_condition,ToolNode
from langchain_community.tools import DuckDuckGoSearchResults

load_dotenv()

True

In [17]:
llm = ChatGroq(model="llama-3.3-70b-versatile", temperature=0)

In [18]:
search_tool = DuckDuckGoSearchResults(max_results=3)

@tool
def calculator(a:float,b:float,operation:str) -> dict:
    """This tool will do basic arithmetic operations on two integers. supported arithmetic operations: add, mul, div, sub"""
    try:
        if operation == "add":
            result = a + b
        elif operation == "sub":
            result = a - b
        elif operation == "mul":
            result = a * b
        elif operation == "div":
            if b == 0:
                return {"error":"division by zero is not allowed"}
            result = a/b
        else:
            return {"error":"Unsupported operation"}

        return {"a":a,"b":b,"operation":operation,"result":result}
    except Exception as e:
        return {"error":str(e)}

ALPHA_VANTAGE_API_KEY = os.environ['ALPHA_VANTAGE_API_KEY']

@tool
def get_stock_price(symbol:str) -> dict:
    """Fetch latest stock price for a given symbol (e.g, 'AAPL','TSLA') using Alpha vantage with API key in the url."""
    url = f'https://www.alphavantage.co/query?function=GLOBAL_QUOTE&symbol={symbol}&apikey={ALPHA_VANTAGE_API_KEY}'
    r = requests.get(url)
    return r.json()
tools = [search_tool,calculator,get_stock_price]
llm_with_tools = llm.bind_tools(tools)

In [19]:

class ChatState(TypedDict):
    messages: Annotated[
        list[BaseMessage], add_messages
    ]  # here we can also use operator.add as reducer, but add_messages is especially optimized for chatbots and is recommended by langgraph.

In [20]:

def chat_node(state: ChatState) -> ChatState:
    """LLM node that may answer or request a tool call"""
    messages = state["messages"]
    res = llm_with_tools.invoke(messages)
    return {"messages": [res]}

tool_node = ToolNode(tools)

In [21]:

graph = StateGraph(ChatState)

graph.add_node("chat_node", chat_node)
graph.add_node("tools", tool_node)

graph.add_edge(START, "chat_node")
graph.add_conditional_edges("chat_node", tools_condition)
graph.add_edge("tools", "chat_node")

chatbot = graph.compile()


In [12]:
res = chatbot.invoke({"messages":[HumanMessage(content="hi")]})
print(res['messages'][-1].content)

I'm here to help. Is there something I can help you with or would you like to chat?


In [14]:
res = chatbot.invoke({"messages":[HumanMessage(content="what is product of 2 and 3?")]})
print(res['messages'][-1].content)

The product of 2 and 3 is 6.


In [23]:
res = chatbot.invoke({"messages":[HumanMessage(content="what is stock price of tesla?")]})
print(res['messages'][-1].content)

The current stock price of Tesla is $313.03.
